In [74]:
print("all ok")


all ok


#### config the model

In [75]:
from langchain_google_genai import ChatGoogleGenerativeAI
model =ChatGoogleGenerativeAI(model='gemini-1.5-flash')
output= model.invoke("hi")
print(output.content)

Hi there! How can I help you today?


### Config the embedding model

In [76]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en")
len(embeddings.embed_query("Hi"))

384

### lets take the data embedded it and store in vdb

In [77]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter


In [78]:
loader = DirectoryLoader("../data2",loader_cls=TextLoader,glob="./*.txt")

In [79]:
docs = loader.load()

In [80]:
docs

[Document(metadata={'source': '../data2/usa.txt'}, page_content="🇺🇸 Overview of the U.S. Economy\nThe United States of America possesses the largest economy in the world in terms of nominal GDP, making it the most powerful economic force globally. It operates under a capitalist mixed economy, where the private sector dominates, but the government plays a significant regulatory and fiscal role. With a population of over 335 million people and a high level of technological advancement, the U.S. economy thrives on a foundation of consumer spending, innovation, global trade, and financial services. It has a highly diversified structure with strong sectors in technology, healthcare, finance, real estate, defense, and agriculture.\n\nU.S. GDP – Size, Composition, and Global Share\nAs of 2024, the United States’ nominal GDP is estimated to be around $28 trillion USD, accounting for approximately 25% of the global economy. It ranks #1 in the world by nominal GDP, far ahead of China (which rank

In [81]:
len(docs)

1

In [82]:
docs[0].page_content

"🇺🇸 Overview of the U.S. Economy\nThe United States of America possesses the largest economy in the world in terms of nominal GDP, making it the most powerful economic force globally. It operates under a capitalist mixed economy, where the private sector dominates, but the government plays a significant regulatory and fiscal role. With a population of over 335 million people and a high level of technological advancement, the U.S. economy thrives on a foundation of consumer spending, innovation, global trade, and financial services. It has a highly diversified structure with strong sectors in technology, healthcare, finance, real estate, defense, and agriculture.\n\nU.S. GDP – Size, Composition, and Global Share\nAs of 2024, the United States’ nominal GDP is estimated to be around $28 trillion USD, accounting for approximately 25% of the global economy. It ranks #1 in the world by nominal GDP, far ahead of China (which ranks 2nd). The U.S. GDP per capita is also among the highest, hover

In [83]:
text_splitter= RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap =50
)

In [84]:
new_docs = text_splitter.split_documents(documents=docs)

In [85]:
new_docs

[Document(metadata={'source': '../data2/usa.txt'}, page_content='🇺🇸 Overview of the U.S. Economy'),
 Document(metadata={'source': '../data2/usa.txt'}, page_content='The United States of America possesses the largest economy in the world in terms of nominal GDP, making it the most powerful economic force globally. It operates under a capitalist mixed economy,'),
 Document(metadata={'source': '../data2/usa.txt'}, page_content='It operates under a capitalist mixed economy, where the private sector dominates, but the government plays a significant regulatory and fiscal role. With a population of over 335 million people and a'),
 Document(metadata={'source': '../data2/usa.txt'}, page_content='a population of over 335 million people and a high level of technological advancement, the U.S. economy thrives on a foundation of consumer spending, innovation, global trade, and financial services.'),
 Document(metadata={'source': '../data2/usa.txt'}, page_content='innovation, global trade, and finan

In [86]:
len(new_docs)

55

In [87]:
 doc_string =[doc.page_content for doc in new_docs]

In [88]:
doc_string

['🇺🇸 Overview of the U.S. Economy',
 'The United States of America possesses the largest economy in the world in terms of nominal GDP, making it the most powerful economic force globally. It operates under a capitalist mixed economy,',
 'It operates under a capitalist mixed economy, where the private sector dominates, but the government plays a significant regulatory and fiscal role. With a population of over 335 million people and a',
 'a population of over 335 million people and a high level of technological advancement, the U.S. economy thrives on a foundation of consumer spending, innovation, global trade, and financial services.',
 'innovation, global trade, and financial services. It has a highly diversified structure with strong sectors in technology, healthcare, finance, real estate, defense, and agriculture.',
 'U.S. GDP – Size, Composition, and Global Share',
 'As of 2024, the United States’ nominal GDP is estimated to be around $28 trillion USD, accounting for approximately 

In [89]:
len(doc_string)

55

In [90]:
db = Chroma.from_documents(documents=new_docs,embedding=embeddings)

In [91]:
retriever = db.as_retriever(search_kwargs={"k":3})

In [92]:
retriever.invoke("industrial growth in usa")

[Document(metadata={'source': '../data2/usa.txt'}, page_content='Looking forward, the U.S. economy is expected to grow at a moderate pace, powered by innovation in AI, green energy, robotics, biotech, and quantum computing. The Biden administration’s Inflation'),
 Document(metadata={'source': '../data2/usa.txt'}, page_content='Looking forward, the U.S. economy is expected to grow at a moderate pace, powered by innovation in AI, green energy, robotics, biotech, and quantum computing. The Biden administration’s Inflation'),
 Document(metadata={'source': '../data2/usa.txt'}, page_content='🇺🇸 Overview of the U.S. Economy')]

### creation of pydantic class

In [93]:

import operator
from pydantic import BaseModel, Field
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage

In [94]:
class TopicSelecctionParser(BaseModel):
    Topic : str = Field(description="selected topic")
    Reasoning: str= Field(description="Reasoning behind the topic selected")

In [119]:
from langchain.output_parsers import PydanticOutputParser  
from langchain_core.output_parsers import StrOutputParser


In [96]:
parser = PydanticOutputParser(pydantic_object=TopicSelecctionParser)

In [97]:
parser.get_format_instructions()


'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"Topic": {"description": "selected topic", "title": "Topic", "type": "string"}, "Reasoning": {"description": "Reasoning behind the topic selected", "title": "Reasoning", "type": "string"}}, "required": ["Topic", "Reasoning"]}\n```'

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"Topic": {"description": "selected topic", "title": "Topic", "type": "string"}, "Reasoning": {"description": "Reasoning behind the topic selected", "title": "Reasoning", "type": "string"}}, "required": ["Topic", "Reasoning"]}\n```'

#### this below agentstate is just for the explnation like how state works


In [98]:
Agentstate ={}

In [99]:
Agentstate['message']=[]

In [100]:
Agentstate

{'message': []}

In [101]:
Agentstate["message"].append("what are you doing")

In [102]:
Agentstate

{'message': ['what are you doing']}

In [103]:
Agentstate["message"].append("HI ")

In [104]:
Agentstate

{'message': ['what are you doing', 'HI ']}

In [105]:
Agentstate['message'][-1]

'HI '

In [106]:
Agentstate['message'][0]

'what are you doing'

#### this agentstate class you need to inside the stategraph

In [107]:
class AgentState(TypedDict):
    messages:Annotated[Sequence[BaseMessage], operator.add]

In [108]:
from langchain.prompts import PromptTemplate

In [109]:
def function_1(state:AgentState):
    question = state['messages'][-1]
    print("question: ",question)

    template = """
                Your task is to classify the given user query into one of the following categories:[USA, Not related].
                Only respond with the category name and nothing else

                User query:{question}
                {format_instructions}
                """
    prompt =PromptTemplate(
        template=template,
        input_variables=['question'],
        partial_variables={'format_instructions':parser.get_format_instructions()}
    )


    chain = prompt | model |parser

    response = chain.invoke({"question":question})

    print(" Parsed Response", response)

    return {"messages":[response.Topic]}

In [110]:
state ={"messages":["what is today's weather"]}

In [111]:
state ={"messages":["what is Gdp of USA"]}

In [112]:
function_1(state)

question:  what is Gdp of USA
 Parsed Response Topic='USA' Reasoning='The query explicitly asks for the GDP of the USA.'


{'messages': ['USA']}

In [115]:
def router(state:AgentState):
    print("-> ROUTER ->")

    last_message = state["messages"][-1]
    print("last_message", last_message)

    if "usa" in last_message.lower():
        return "RAG Call"
    else:
        return "LLM Call"

In [116]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [139]:
from langchain_core.runnables import RunnablePassthrough

In [140]:
# RAG Function
def function_2(state:AgentState):
    question = state['messages'][0]

    prompt = PromptTemplate(
        template="""You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:""",
        
        input_variables=['context', 'question']
    )
    
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | model
        | StrOutputParser()
    )
    result = rag_chain.invoke(question)
    
    return {"messages":[result]}

In [ ]:
# LLM 
def function_3(state:AgentState):
    print("-.LLM Call ->")
    question = state["messages"][0]

    #normal llm call
    complete_query ="Answer the follow question with you knowledge of the real world. Following is the user question: "+question
    response= model.invoke(complete_query)

    return {"messages":[response.content]}
    

In [142]:
from langgraph.graph import StateGraph, END

In [143]:
workflow = StateGraph(AgentState)

In [144]:
workflow.add_node("Supervisor",function_1)

In [145]:
workflow.add_node("RAG",function_2)

In [146]:
workflow.add_node("LLM", function_3)

In [147]:
workflow.set_entry_point("Supervisor")

In [150]:
workflow.add_conditional_edges(
    "Supervisor",
    router,
    { 
        "RAG Call":"RAG",
        "LLM Call":"LLM"

    }
)

ValueError: Branch with name `router` already exists for node `Supervisor`

In [151]:
workflow.add_edge("RAG",END)
workflow.add_edge("LLM",END)

In [152]:
app =workflow.compile()

In [153]:
state = {"messages":["Hi"]}

In [154]:
app.invoke(state)

question:  Hi
 Parsed Response Topic='Not related' Reasoning="The query 'Hi' is a greeting and does not contain any information related to the USA."
-> ROUTER ->
last_message Not related
-.LLM Call ->


{'messages': ['Hi', 'Not related', 'Hi!  How can I help you today?']}

In [155]:
state={"messages":["what is a gdp of usa?"]}

In [156]:
app.invoke(state)


question:  what is a gdp of usa?
 Parsed Response Topic='USA' Reasoning='The query explicitly asks for the GDP of the USA.'
-> ROUTER ->
last_message USA


{'messages': ['what is a gdp of usa?',
  'USA',
  "The U.S. nominal GDP is approximately $28 trillion USD as of 2024.  This represents about 25% of the global economy.  It's the largest nominal GDP globally."]}

In [157]:
state={"messages":["can you tell me the industrial growth of world's most powerful economy?"]}

In [158]:
result = app.invoke(state)

question:  can you tell me the industrial growth of world's most powerful economy?
 Parsed Response Topic='USA' Reasoning="The query asks about the industrial growth of the world's most powerful economy, which is generally considered to be the United States."
-> ROUTER ->
last_message USA


In [159]:
result["messages"][-1]

"The U.S. has the world's largest economy, with a nominal GDP of $28 trillion.  It's considered the engine of global growth due to innovation, financial strength, and its institutional framework.  I don't have specific industrial growth figures readily available."